# 10a — Binary Error Analysis

This notebook analyzes:
- Ben-Sarc binary: BanglaBERT vs BanglaBERT+FGM
- BanglaSarc3-binary: BanglaBERT vs BanglaBERT+FGM

It generates:
- top confident errors
- cases improved by FGM
- cases degraded by FGM

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import confusion_matrix

TABLES = Path("../04_outputs/tables")
SPLITS = Path("../01_data/interim/splits")
CHECKPOINTS = Path("../03_models/checkpoints")
MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def resolve_checkpoint_dir(checkpoint_root):
    checkpoint_root = Path(checkpoint_root)
    if (checkpoint_root / "config.json").exists():
        return checkpoint_root
    ckpts = sorted(
        [p for p in checkpoint_root.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: int(p.name.split("-")[-1])
    )
    if not ckpts:
        raise FileNotFoundError(f"No checkpoint-* folder found inside: {checkpoint_root}")
    trainer_state_file = checkpoint_root / "trainer_state.json"
    if trainer_state_file.exists():
        with open(trainer_state_file, "r", encoding="utf-8") as f:
            trainer_state = json.load(f)
        best_ckpt = trainer_state.get("best_model_checkpoint", None)
        if best_ckpt:
            best_ckpt = Path(best_ckpt)
            if best_ckpt.exists():
                return best_ckpt
    return ckpts[-1]

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_binary_predictions(checkpoint_root, split_file):
    df = pd.read_csv(split_file).copy()
    ds_df = df[["text", "label_binary"]].rename(columns={"label_binary": "label"})
    ds = Dataset.from_pandas(ds_df, preserve_index=False)
    ds = ds.map(tokenize_batch, batched=True)
    ds = ds.remove_columns(["text"])
    ds.set_format("torch")

    checkpoint_dir = resolve_checkpoint_dir(checkpoint_root)
    print("Loading:", checkpoint_dir)

    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
    trainer = Trainer(model=model)
    output = trainer.predict(ds)
    logits = output.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    out = df.copy()
    out["y_true"] = df["label_binary"].astype(int).to_numpy()
    out["y_pred"] = preds
    out["prob_0"] = probs[:, 0]
    out["prob_1"] = probs[:, 1]
    out["correct"] = out["y_true"] == out["y_pred"]
    out["pred_conf"] = np.where(out["y_pred"] == 1, out["prob_1"], out["prob_0"])
    return out

In [3]:
def top_errors(df, n=20):
    return df[df["correct"] == False].sort_values("pred_conf", ascending=False)[
        ["text", "y_true", "y_pred", "prob_0", "prob_1", "pred_conf"]
    ].head(n)

def compare_models(base_df, improved_df, n=20):
    merged = pd.DataFrame({
        "text": base_df["text"],
        "y_true": base_df["y_true"],
        "base_pred": base_df["y_pred"],
        "base_correct": base_df["correct"],
        "base_conf": base_df["pred_conf"],
        "improved_pred": improved_df["y_pred"],
        "improved_correct": improved_df["correct"],
        "improved_conf": improved_df["pred_conf"],
    })
    improved_cases = merged[(merged["base_correct"] == False) & (merged["improved_correct"] == True)].copy()
    degraded_cases = merged[(merged["base_correct"] == True) & (merged["improved_correct"] == False)].copy()
    return improved_cases.head(n), degraded_cases.head(n)

In [4]:
# Ben-Sarc
ben_plain = load_binary_predictions(
    "../03_models/checkpoints/banglabert_ben_sarc_binary",
    "../01_data/interim/splits/ben_sarc_binary_test.csv",
)
ben_fgm = load_binary_predictions(
    "../03_models/checkpoints/banglabert_fgm_ben_sarc_binary",
    "../01_data/interim/splits/ben_sarc_binary_test.csv",
)

print("Ben-Sarc plain confusion matrix:")
print(confusion_matrix(ben_plain["y_true"], ben_plain["y_pred"]))
print("Ben-Sarc FGM confusion matrix:")
print(confusion_matrix(ben_fgm["y_true"], ben_fgm["y_pred"]))

Map: 100%|██████████| 2564/2564 [00:00<00:00, 14081.48 examples/s]


Loading: ../03_models/checkpoints/banglabert_ben_sarc_binary/checkpoint-5128


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8917.07it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Map: 100%|██████████| 2564/2564 [00:00<00:00, 21238.53 examples/s]


Loading: ../03_models/checkpoints/banglabert_fgm_ben_sarc_binary/checkpoint-5128


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9248.99it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Ben-Sarc plain confusion matrix:
[[1095  187]
 [ 335  947]]
Ben-Sarc FGM confusion matrix:
[[1061  221]
 [ 267 1015]]


In [5]:
print("Top Ben-Sarc errors — plain")
display(top_errors(ben_plain, n=20))

print("Top Ben-Sarc errors — FGM")
display(top_errors(ben_fgm, n=20))

ben_improved, ben_degraded = compare_models(ben_plain, ben_fgm, n=20)

print("Ben-Sarc cases fixed by FGM")
display(ben_improved)

print("Ben-Sarc cases worsened by FGM")
display(ben_degraded)

Top Ben-Sarc errors — plain


,text,y_true,y_pred,prob_0,prob_1,pred_conf
1274,আপনারা ফেসবুকে অনেকেই অনেক কিছু বলেন বা লিখেন ...,1,0,0.992626,0.007373,0.992626
817,চোটের কারণে দলের বাইরে চলে গেছেন এবি ডি ভিলিয়...,1,0,0.992159,0.007841,0.992159
991,অসম্ভব ভাল ছবি । এক কথায় অনবদ্য । বহুদিন পরে ...,1,0,0.992047,0.007953,0.992047
30,বাংলাদেশ জাতীয় দলের ভবিষ্যৎ কান্ডারি বলে মনে হয় ।,1,0,0.991984,0.008016,0.991984
1198,একটু দেরি হওয়াতে মৃত্যুর মুখে পতিত হচ্ছে বা হব...,1,0,0.991953,0.008047,0.991953
2043,অনন্ত জলিলের সাথে ছবি দেখতে চাই,1,0,0.991800,0.008200,0.991800
2533,বেশ কিছুদিন পরে হলেও আবারও সেই কাঙ্ক্ষিত সোনার...,1,0,0.991768,0.008232,0.991768
2505,ছোট বেলায় আম্মু ঈদের সালামির টাকা সব নিয়া যা...,1,0,0.991457,0.008543,0.991457
1259,রকেট গুলো আরো নিখুঁত ভাবে মারতে হবে ।,1,0,0.991429,0.008571,0.991429
399,আর এদিকে সময়ের অন্যতম সেরা দল জিম্বাবুয়েকে তাদ...,1,0,0.991368,0.008632,0.991368


Top Ben-Sarc errors — FGM


,text,y_true,y_pred,prob_0,prob_1,pred_conf
561,ভালা লাগছে তুমি তাদের মনে রেখছো বলে,1,0,0.986784,0.013216,0.986784
817,চোটের কারণে দলের বাইরে চলে গেছেন এবি ডি ভিলিয়...,1,0,0.985418,0.014582,0.985418
1259,রকেট গুলো আরো নিখুঁত ভাবে মারতে হবে ।,1,0,0.984402,0.015598,0.984402
991,অসম্ভব ভাল ছবি । এক কথায় অনবদ্য । বহুদিন পরে ...,1,0,0.984258,0.015742,0.984258
2210,ইয়ার্কির একটা সীমা থাকা উচিৎ,1,0,0.983716,0.016284,0.983716
2506,সাকিব সারাবছর ডাক মারলেও তবু একদল এলিয়েন বলবে ...,1,0,0.982999,0.017001,0.982999
1202,ঈদ শুধু আপনাদের জন্যই !,1,0,0.981499,0.018501,0.981499
2533,বেশ কিছুদিন পরে হলেও আবারও সেই কাঙ্ক্ষিত সোনার...,1,0,0.980683,0.019317,0.980683
1834,এতে আশ্চর্যের কি আছে ! নগর পিতা যদি কয়েক কোটি ...,1,0,0.979713,0.020287,0.979713
2454,ঊনিশশো বাইশ এর পর ঊনিশশো তেইশ থেকে শুরু করে ঊন...,1,0,0.979540,0.020460,0.979540


Ben-Sarc cases fixed by FGM


,text,y_true,base_pred,base_correct,base_conf,improved_pred,improved_correct,improved_conf
27,আমি রিজেক্ট করে দিছি ! এখন অন্যকাউকে খুঁজতে কত...,1,0,False,0.778644,1,True,0.575915
88,ঈদে কোথাও ঘুড়তে না গিয়ে নোভেল এর পেইজ ঘুরতেছি,1,0,False,0.947472,1,True,0.711197
89,এতো ফালতু আবেগে কেঁদে ফেলছি ভাই ।,1,0,False,0.746297,1,True,0.590879
91,এখানে একদল চলে এসেছে কান্নার রিয়াক্ট দিচ্ছে ।...,1,0,False,0.794051,1,True,0.695968
116,ফুলসজ্জার রাতে আবার কীসের ঝড়,1,0,False,0.661214,1,True,0.562409
149,সাকিব ভাই মদিনাতে কবে আসলেন । ভাই যদি জানতাম য...,1,0,False,0.515210,1,True,0.552351
153,সব খোলা সুধু কোর্ট আর শিক্ষাপ্রতিষ্ঠান ই বন্ধ ।,0,1,False,0.846647,0,True,0.537576
171,তুমি তো এত ইংরেজি পারো না । পিকক্ কে বলো ককপিট...,1,0,False,0.960410,1,True,0.669946
187,কেউ বলতে পারেন পা এই ভাবে ব্যাকা রাখার রহস্যটা...,1,0,False,0.562383,1,True,0.606828
220,ফেরি বাড়িয়ে দেওয়া হোক । পরিবহন ডাবল করে দেওয়া ...,0,1,False,0.813744,0,True,0.721972


Ben-Sarc cases worsened by FGM


,text,y_true,base_pred,base_correct,base_conf,improved_pred,improved_correct,improved_conf
36,মুখে মধু অন্তরে বিষ ।,0,0,True,0.768625,1,False,0.575827
42,ছেলেটা প্রিতভাবান ব্যাটসম্যান !,0,0,True,0.917910,1,False,0.582879
70,আমার বাসায় হার্ডবোর্ডের একটা সিঙ্গেল রুম ফাঁক...,0,0,True,0.983264,1,False,0.582775
86,টাংগাইলের কেউ যদি এসপিসি একাউন্ট করেন আমাকে জা...,0,0,True,0.830560,1,False,0.500860
117,মনেহয় সে যুদ্ধ না কইরা চ্যাগাইয়া শুইয়া থাকতো ।,1,1,True,0.881231,0,False,0.611280
132,সুরের পাখি মাহফুজুর রহমানের একক শিষ্য ।,1,1,True,0.832220,0,False,0.521546
133,সব জায়গায় নষ্টামি । কি দরকার আরেকজনের পেটে লাথ...,0,0,True,0.907347,1,False,0.689025
138,রোম যখন পুড়ছিলো নিরো তখন ডাকছিলো দিদি ও দিদিই ।,0,0,True,0.933950,1,False,0.785426
147,এরপর আর কি বলবো আপনাদের মন ই ভারত কেন্দ্রিক ।,0,0,True,0.759823,1,False,0.601782
182,প্রয়োজন হলে গাছ কাটবে অন্যদিকে প্রয়োজনমত গাছ ল...,0,0,True,0.629060,1,False,0.751093


In [6]:
# BanglaSarc3-binary
bs3_plain = load_binary_predictions(
    "../03_models/checkpoints/banglabert_banglasarc3_binary",
    "../01_data/interim/splits/banglasarc3_binary_test.csv",
)
bs3_fgm = load_binary_predictions(
    "../03_models/checkpoints/banglabert_fgm_banglasarc3_binary",
    "../01_data/interim/splits/banglasarc3_binary_test.csv",
)

print("BanglaSarc3-binary plain confusion matrix:")
print(confusion_matrix(bs3_plain["y_true"], bs3_plain["y_pred"]))
print("BanglaSarc3-binary FGM confusion matrix:")
print(confusion_matrix(bs3_fgm["y_true"], bs3_fgm["y_pred"]))

Map: 100%|██████████| 802/802 [00:00<00:00, 13189.01 examples/s]


Loading: ../03_models/checkpoints/banglabert_banglasarc3_binary/checkpoint-1604


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7431.86it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Map: 100%|██████████| 802/802 [00:00<00:00, 14779.32 examples/s]


Loading: ../03_models/checkpoints/banglabert_fgm_banglasarc3_binary/checkpoint-1604


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7372.39it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


BanglaSarc3-binary plain confusion matrix:
[[309  92]
 [116 285]]
BanglaSarc3-binary FGM confusion matrix:
[[288 113]
 [ 91 310]]


In [7]:
print("Top BanglaSarc3-binary errors — plain")
display(top_errors(bs3_plain, n=20))

print("Top BanglaSarc3-binary errors — FGM")
display(top_errors(bs3_fgm, n=20))

bs3_improved, bs3_degraded = compare_models(bs3_plain, bs3_fgm, n=20)

print("BanglaSarc3-binary cases fixed by FGM")
display(bs3_improved)

print("BanglaSarc3-binary cases worsened by FGM")
display(bs3_degraded)

Top BanglaSarc3-binary errors — plain


,text,y_true,y_pred,prob_0,prob_1,pred_conf
247,মনোযোগ দিয়ে প্রতিটা কথা শুনুন এবং সে অনুযায়ী...,1,0,0.975902,0.024098,0.975902
695,ইতিহাসের পাতায় লেখা থাকবে ৭১এর পরাজিত শক্তি মা...,1,0,0.969992,0.030008,0.969992
159,ধর্ম গ্রন্থের রেফারেন্স টানার আগে ঘটনার প্রেক্...,1,0,0.969643,0.030358,0.969643
427,চুপ কর সবাই পাপীর দল বাচ্চাডা ভালুপাশা প্রকাশ ...,1,0,0.968527,0.031473,0.968527
314,এটাই বাংলাদেশের সাথে সুইজারল্যান্ডের পার্থক্য ...,1,0,0.966693,0.033307,0.966693
136,এইগুলারে আর এদের মা বাবাদের চিরিয়াখানায় রাখা উচিৎ,1,0,0.963026,0.036974,0.963026
195,যে লিখেছে সে আসলেই ভালো লিখেছে বিশ্বাস না হলে ...,1,0,0.962894,0.037106,0.962894
158,ধন্যবাদ নিজের পরিচয় দেওয়ার জন্য,1,0,0.961295,0.038705,0.961295
570,যে যেই যায়গার লোক তার টার্গেট তো সেখানেই থাকবে,1,0,0.960760,0.039240,0.960760
553,আপনার পোষ্ট গুলো অসাধারণ সবসময় সত্য কথা তুলে ধ...,1,0,0.960043,0.039957,0.960043


Top BanglaSarc3-binary errors — FGM


,text,y_true,y_pred,prob_0,prob_1,pred_conf
159,ধর্ম গ্রন্থের রেফারেন্স টানার আগে ঘটনার প্রেক্...,1,0,0.960469,0.039531,0.960469
553,আপনার পোষ্ট গুলো অসাধারণ সবসময় সত্য কথা তুলে ধ...,1,0,0.955068,0.044932,0.955068
247,মনোযোগ দিয়ে প্রতিটা কথা শুনুন এবং সে অনুযায়ী...,1,0,0.949347,0.050653,0.949347
570,যে যেই যায়গার লোক তার টার্গেট তো সেখানেই থাকবে,1,0,0.942888,0.057112,0.942888
461,হ্যাতের ভাষণ দুনিয়ার সবচেয়ে বিরক্তিকর জিনিস রি...,1,0,0.942541,0.057459,0.942541
136,এইগুলারে আর এদের মা বাবাদের চিরিয়াখানায় রাখা উচিৎ,1,0,0.932944,0.067056,0.932944
161,আল্লাহ সকল মীমের মনের আশা পূরণ করুন আমিন,1,0,0.932072,0.067928,0.932072
695,ইতিহাসের পাতায় লেখা থাকবে ৭১এর পরাজিত শক্তি মা...,1,0,0.924679,0.075321,0.924679
325,১৬ বছর পর প্রভু চেঞ্জ হইছে আপনার ১৬ বছরের অভ্য...,1,0,0.924294,0.075706,0.924294
575,আগে ছাত্র সমাজকে কোন পাত্তাই দিছেন না এখন তাদে...,1,0,0.922360,0.077640,0.922360


BanglaSarc3-binary cases fixed by FGM


,text,y_true,base_pred,base_correct,base_conf,improved_pred,improved_correct,improved_conf
17,স্বামীর জন্য কান্নার যুগ শেষ হতে চললো,1,0,False,0.700813,1,True,0.609445
44,আল্লায় বাচাইছে এর পোষাক লাগবে এমন যাতে গাড়ি ...,1,0,False,0.640225,1,True,0.580518
45,বাংলাদেশে সবাই ওর বাপের সালা,1,0,False,0.763994,1,True,0.629540
63,ইউনুস ও কি তাহলে হাসিনার মতো গুম খুন লুট শুরু ...,1,0,False,0.781374,1,True,0.598475
98,আমি একজনকে চিনি যে আম্লীগ ক্ষমতায় আসুক বা বিএ...,1,0,False,0.744262,1,True,0.537427
114,ওই হালা বিয়ে করুক,1,0,False,0.596867,1,True,0.792071
123,এই মূহুর্তে দরকার আওয়ামী লীগের সরকার,1,0,False,0.755383,1,True,0.732039
124,এইডা কোনো কথা হইলো,1,0,False,0.647052,1,True,0.505212
137,বিমান ভাড়া বাংলাদেশের মতো এত ডাকাতি কোনো দেশে...,0,1,False,0.703983,0,True,0.799077
147,কি কমু ভাই আজকে যা দেখছি মাথাই নষ্ট,1,0,False,0.894597,1,True,0.674120


BanglaSarc3-binary cases worsened by FGM


,text,y_true,base_pred,base_correct,base_conf,improved_pred,improved_correct,improved_conf
14,ট্রাস্ট মি ৯০ ঝগড়া ঝগড়ী কমে যাবে লাইফটা অনেক স...,1,1,True,0.872065,0,False,0.629569
27,যারা সমালোচনা করতেছে তাদের মানিব্যাগ ঘাটলে ৯০০...,0,0,True,0.523861,1,False,0.582323
64,যে বউকে ভয় পায় সে সেরা পুরুষ,1,1,True,0.801007,0,False,0.656066
86,কান্নাকাটির কর্মসূচি কেন সেক্স এর কর্মসূচি দে ...,0,0,True,0.843051,1,False,0.508889
88,স্বাদ আছে কিন্তু স্বাধ্য নাই,0,0,True,0.714744,1,False,0.661319
140,এটা দিয়ে কি করা যাবে,0,0,True,0.863382,1,False,0.671652
160,আমার এই ফোনটা কিনার অনেক শখ ছিলো,0,0,True,0.611815,1,False,0.733254
178,একটু ভালো ইংরেজি কইতে পারে বইলা এক্টা বাচ্চা ছ...,0,0,True,0.536405,1,False,0.839875
181,লোকে সবসময় আমার অক্ষমতার কথা মনে করিয়ে দেয় হায় রে,0,0,True,0.809849,1,False,0.639086
191,যাদের এখনো সন্দেহ রয়েছে তারা একটু পানি পান করে...,1,1,True,0.818285,0,False,0.563000


In [8]:
summary = pd.DataFrame([
    {"dataset": "ben_sarc_binary", "plain_errors": int((~ben_plain["correct"]).sum()), "fgm_errors": int((~ben_fgm["correct"]).sum())},
    {"dataset": "banglasarc3_binary", "plain_errors": int((~bs3_plain["correct"]).sum()), "fgm_errors": int((~bs3_fgm["correct"]).sum())},
])
summary["error_reduction"] = summary["plain_errors"] - summary["fgm_errors"]
display(summary)
summary.to_csv(TABLES / "binary_error_analysis_summary.csv", index=False)
print("Saved:", TABLES / "binary_error_analysis_summary.csv")

,dataset,plain_errors,fgm_errors,error_reduction
0,ben_sarc_binary,522,488,34
1,banglasarc3_binary,208,204,4


Saved: ../04_outputs/tables/binary_error_analysis_summary.csv
